In [1]:
import pandas as pd
import random
from transformers import AutoTokenizer
from utils.load import LoadDataCSV
from utils.embeddings import Embeddings

# Load datasets

df_loader=LoadDataCSV()


#  Data Preparation

In [6]:
# Import pre-split data
fact_checks_train=pd.read_csv('data/transformed/fact_checks_train.csv')
posts_train=pd.read_csv('data/transformed/posts_train_train.csv')
fact_checks_val=pd.read_csv('data/transformed/fact_checks_val.csv')
posts_val=pd.read_csv('data/transformed/posts_train_val.csv')

pairs_train=pd.read_csv('data/transformed/pairs_train.csv')
pairs_val=pd.read_csv('data/transformed/pairs_val.csv')
pairs=pd.read_csv('data/transformed/pairs.csv')


In [2]:
fact_checks_mono_sample_1000=pd.read_csv('data/fact_checks_mono_sample_1000.csv')
posts_mono_sample_1000=pd.read_csv('data/posts_train_mono_sample_1000.csv')


In [3]:

embeddings_utils=Embeddings

embedding_columns_posts = ['ocr_text_embedding', 'ocr_text_eng_embedding']
embedding_columns_fact_checks = ['claim_title_eng_embedding', 'claim_title_embedding']

fact_checks_mono_sample_1000=embeddings_utils.restore_embeddings(fact_checks_mono_sample_1000, embedding_columns_fact_checks)
posts_mono_sample_1000=embeddings_utils.restore_embeddings(posts_mono_sample_1000, embedding_columns_posts)

In [4]:
# Load your data

df_fact_checks, df_posts, df_fact_check_post_mapping = df_loader.load_data()


In [5]:

monolingual_data= df_loader.split_data()

fact_checks_mono = monolingual_data["fact_checks"]
posts_train_mono = monolingual_data["posts_train"]
posts_dev_mono = monolingual_data["posts_dev"]

TypeError: LoadDataCSV.combine_claim_text() takes 1 positional argument but 2 were given

In [4]:

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-large-instruct")


In [7]:
#df_posts=posts_mono_sample_1000.copy()
#df_fact_checks=fact_checks_mono_sample_1000.copy()


In [8]:
# Corrected logic for combining OCR and text into ocr_text column
def combine_ocr_text(row):
    ocr = row['ocr']
    text = row['text']
    if ocr and isinstance(ocr, list) and len(ocr) > 0 and len(ocr[0]) > 0:
        return 'passage: '+ str(ocr[0][0]) + ' ' + (str(text[0]) if text else '')
    elif text:
        return 'passage: '+ str(text[0])
    else:
        try:
            return 'query: '+ str(ocr[0]) + ' ' + (str(text[0]) if text else '')
        except:
            return ''

# Apply the function to create the new column
df_posts['ocr_text'] = df_posts.apply(combine_ocr_text, axis=1)

# Corrected logic for combining OCR and text into ocr_text column
def combine_claim_text(row):
    claim = row['claim']
    title = row['title']
    if claim and isinstance(claim, list) and len(claim) > 0 and len(claim[0]) > 0:
        return 'query: '+ str(claim[0][0]) + ' ' + (str(title[0]) if title else '')
    elif title:
        return 'query: '+ str(title[0])
        try:
            return 'query: '+ str(claim[0]) + ' ' + (str(title[0]) if title else '')
        except:
            return ''

# Apply the function to create the new column
df_fact_checks['claim_title'] = df_fact_checks.apply(combine_claim_text, axis=1)

# Delete outliers

In [9]:
# Calcula la longitud del texto en ocr_text
df_posts["text_length"] = df_posts["ocr_text"].str.split().apply(len)

# Obtén los post_id con longitud de texto mayor a 1000
long_text_post_ids = df_posts[df_posts["text_length"] > 1000].index

# Filtra df_fact_check_post_mapping para eliminar esos post_id
df_fact_check_post_mapping_filtered = df_fact_check_post_mapping[
    ~df_fact_check_post_mapping["post_id"].isin(long_text_post_ids)
]

# Verifica el resultado
len(df_fact_check_post_mapping_filtered), len(df_fact_check_post_mapping)

(25704, 25743)

In [10]:
df_posts['post_id']=df_posts.index
df_fact_checks['fact_check_id']=df_fact_checks.index

In [11]:
# Create a df of mapping for post_id that show up in posts_mono_sample_1000
df_posts_filtered=df_posts[df_posts['post_id'].isin(df_fact_check_post_mapping_filtered['post_id'])]

df_fact_checks_filtered=df_fact_checks[df_fact_checks['fact_check_id'].isin(df_fact_check_post_mapping_filtered['fact_check_id'])]


In [12]:
len(df_posts_filtered.index.unique()), len(df_fact_checks_filtered.index.unique())

(21950, 14205)

# Small samples

In [13]:
# Create a df of mapping for post_id that show up in posts_mono_sample_1000
mapping_sample_1000=df_fact_check_post_mapping[df_fact_check_post_mapping['post_id'].isin(posts_mono_sample_1000['post_id'])]

In [14]:
# Generate extremely small samples
posts_mono_sample_1000_10=posts_mono_sample_1000.sample(n=10)

# Create a df of mapping for post_id that show up in posts_mono_sample_1000
mapping_sample_1000_10=df_fact_check_post_mapping[df_fact_check_post_mapping['post_id'].isin(posts_mono_sample_1000_10['post_id'])]

fact_checks_mono_sample_1000_10=fact_checks_mono_sample_1000[fact_checks_mono_sample_1000['fact_check_id'].isin(mapping_sample_1000_10['fact_check_id'])]


# Split into chunks

In [5]:

def split_into_chunks(text, max_tokens=512):
    """
    Divide un texto en chunks de hasta max_tokens tokens.
    """
    if not text or not isinstance(text, str):  # Verifica que sea una cadena de texto
        return []
    tokens = tokenizer.encode(text, truncation=True)
    chunks = [tokens[i:i + max_tokens] for i in range(0, len(tokens), max_tokens)]
    return [tokenizer.decode(chunk, skip_special_tokens=True) for chunk in chunks]



def create_pairs(pairs_df, neg_ratio=5):
    """
    Creates positive and negative pairs for contrastive learning.
    """
    positive_pairs = []
    negative_pairs = []

    all_fact_check_ids = set(pairs_df["fact_check_id"].tolist())
    for _, row in pairs_df.iterrows():
        # Positive pair
        positive_pairs.append((row["post_id"], row["fact_check_id"], 1))
        
        # Negative pairs
        possible_negatives = list(all_fact_check_ids - {row["fact_check_id"]})
        if len(possible_negatives) < neg_ratio:
            neg_ratio = len(possible_negatives)  # Adjust if not enough negatives
        negatives = random.sample(possible_negatives, neg_ratio)
        for neg in negatives:
            negative_pairs.append((row["post_id"], neg, 0))
    
    return positive_pairs + negative_pairs



In [16]:
# Chunk posts and fact-check texts
posts_mono_sample_1000["chunks"] = posts_mono_sample_1000["ocr_text"].apply(split_into_chunks)

fact_checks_mono_sample_1000["chunks"] = fact_checks_mono_sample_1000["claim_title"].apply(split_into_chunks)

pairs_data_1000 = create_pairs(mapping_sample_1000)

In [17]:
posts_mono_sample_1000_10["chunks"] = posts_mono_sample_1000_10["ocr_text"].apply(split_into_chunks)

fact_checks_mono_sample_1000_10["chunks"] = fact_checks_mono_sample_1000_10["claim_title"].apply(split_into_chunks)

pairs_data_10 = create_pairs(mapping_sample_1000_10)

C:\Users\nicol\AppData\Local\Temp\ipykernel_17876\3193510769.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fact_checks_mono_sample_1000_10["chunks"] = fact_checks_mono_sample_1000_10["claim_title"].apply(split_into_chunks)


In [18]:
# Chunk posts and fact-check texts
df_posts_filtered["chunks"] = df_posts_filtered["ocr_text"].apply(split_into_chunks)

df_fact_checks_filtered["chunks"] = df_fact_checks_filtered["claim_title"].apply(split_into_chunks)

pairs_data_filtered = create_pairs(df_fact_check_post_mapping_filtered)

C:\Users\nicol\AppData\Local\Temp\ipykernel_17876\4025107908.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_posts_filtered["chunks"] = df_posts_filtered["ocr_text"].apply(split_into_chunks)
C:\Users\nicol\AppData\Local\Temp\ipykernel_17876\4025107908.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fact_checks_filtered["chunks"] = df_fact_checks_filtered["claim_title"].apply(split_into_chunks)


In [12]:
# Pre-split data
df_posts_filtered_train=posts_train.copy()
df_fact_checks_filtered_train=fact_checks_train.copy()
df_posts_filtered_val=posts_val.copy()
df_fact_checks_filtered_val=fact_checks_val.copy()

df_posts_filtered_train["chunks"] = posts_train["text_ocr_stripped"].apply(split_into_chunks)

df_fact_checks_filtered_train["chunks"] = fact_checks_train["claim_title_stripped"].apply(split_into_chunks)

df_posts_filtered_val["chunks"] = posts_val["text_ocr_stripped"].apply(split_into_chunks)

df_fact_checks_filtered_val["chunks"] = fact_checks_val["claim_title_stripped"].apply(split_into_chunks)    

pairs_data_filtered_train = create_pairs(pairs_train)
pairs_data_filtered_val = create_pairs(pairs_val)

# Split into train and dev

In [19]:
# Split into training and validation sets
from sklearn.model_selection import train_test_split

#posts_train_mono= posts_train_mono.sample(n=1000, random_state=42)

posts_train, posts_val = train_test_split(posts_train_mono, test_size=0.2, random_state=42)

In [20]:
df_fact_check_post_mapping_filtered_train=df_fact_check_post_mapping_filtered[df_fact_check_post_mapping_filtered['post_id'].isin(posts_train['post_id'])]
df_fact_check_post_mapping_filtered_val=df_fact_check_post_mapping_filtered[df_fact_check_post_mapping_filtered['post_id'].isin(posts_val['post_id'])]

In [21]:
fact_checks_train=fact_checks_mono[fact_checks_mono['fact_check_id'].isin(df_fact_check_post_mapping_filtered_train['fact_check_id'])]
fact_checks_val=fact_checks_mono[fact_checks_mono['fact_check_id'].isin(df_fact_check_post_mapping_filtered_val['fact_check_id'])]

In [22]:
df_posts_filtered_train=df_posts_filtered[df_posts_filtered['post_id'].isin(posts_train['post_id'])]
df_posts_filtered_val=df_posts_filtered[df_posts_filtered['post_id'].isin(posts_val['post_id'])]

df_fact_checks_filtered_train=df_fact_checks_filtered[df_fact_checks_filtered['fact_check_id'].isin(fact_checks_train['fact_check_id'])]
df_fact_checks_filtered_val=df_fact_checks_filtered[df_fact_checks_filtered['fact_check_id'].isin(fact_checks_val['fact_check_id'])]

pairs_data_filtered_train = create_pairs(df_fact_check_post_mapping_filtered_train)
pairs_data_filtered_val = create_pairs(df_fact_check_post_mapping_filtered_val)

In [23]:
post_ids_in_data = set(pair[0] for pair in pairs_data_filtered)
post_ids_in_df = set(df_posts_filtered.index)

fact_check_ids_in_data = set(pair[1] for pair in pairs_data_filtered)
fact_check_ids_in_df = set(df_fact_checks_filtered.index)

missing_post_ids = post_ids_in_data - post_ids_in_df
missing_fact_check_ids = fact_check_ids_in_data - fact_check_ids_in_df

print("Missing post_ids:", missing_post_ids)
print("Missing fact_check_ids:", missing_fact_check_ids)


Missing post_ids: set()
Missing fact_check_ids: set()


In [24]:
post_ids_in_data = set(pair[0] for pair in pairs_data_filtered_train)
post_ids_in_df = set(df_posts_filtered_train.index)

fact_check_ids_in_data = set(pair[1] for pair in pairs_data_filtered_train)
fact_check_ids_in_df = set(df_fact_checks_filtered_train.index)

missing_post_ids = post_ids_in_data - post_ids_in_df
missing_fact_check_ids = fact_check_ids_in_data - fact_check_ids_in_df

print("Missing post_ids:", missing_post_ids)
print("Missing fact_check_ids:", missing_fact_check_ids)


Missing post_ids: set()
Missing fact_check_ids: set()


# Fine-Tuning Dataset Class

In [10]:
import torch
from torch.utils.data import Dataset
import random

class FineTuneContrastiveDataset(Dataset):
    def __init__(self, pairs_data, posts, fact_checks, tokenizer, max_tokens=512):
        """
        pairs_data: Lista de tuplas (post_id, fact_check_id, label).
        posts: DataFrame con los textos asociados a los post_id.
        fact_checks: DataFrame con los textos asociados a los fact_check_id.
        tokenizer: Tokenizador de Hugging Face.
        max_tokens: Máxima longitud permitida de los textos.
        """
        self.pairs_data = pairs_data
        self.posts = posts.set_index("post_id")
        self.fact_checks = fact_checks.set_index("fact_check_id")
        self.tokenizer = tokenizer
        self.max_tokens = max_tokens
        self.fact_check_ids = list(self.fact_checks.index)  # Para seleccionar negativos

    def __len__(self):
        return len(self.pairs_data)

    def __getitem__(self, idx):
        # Recupera datos del par original
        post_id, fact_check_id, label = self.pairs_data[idx]
        
        # Texto del post (anchor)
        post_text = " ".join(self.posts.loc[post_id]["chunks"])

        # Texto del fact-check positivo (positive)
        fact_check_text_positive = " ".join(self.fact_checks.loc[fact_check_id]["chunks"])

        # Genera un fact-check negativo (negative)
        negative_fact_check_id = random.choice(
            [fc_id for fc_id in self.fact_check_ids if fc_id != fact_check_id]
        )
        fact_check_text_negative = " ".join(self.fact_checks.loc[negative_fact_check_id]["chunks"])


        # Tokenización
        inputs_anchor = self.tokenizer(post_text, 
                                       truncation=True, max_length=self.max_tokens, 
                                       padding="max_length", return_tensors="pt")
        inputs_positive = self.tokenizer(fact_check_text_positive, 
                                         truncation=True, max_length=self.max_tokens, 
                                         padding="max_length", return_tensors="pt")
        inputs_negative = self.tokenizer(fact_check_text_negative, 
                                         truncation=True, max_length=self.max_tokens, 
                                         padding="max_length", return_tensors="pt")

        return {
            "anchor_input_ids": inputs_anchor["input_ids"].squeeze(0),
            "anchor_attention_mask": inputs_anchor["attention_mask"].squeeze(0),
            "positive_input_ids": inputs_positive["input_ids"].squeeze(0),
            "positive_attention_mask": inputs_positive["attention_mask"].squeeze(0),
            "negative_input_ids": inputs_negative["input_ids"].squeeze(0),
            "negative_attention_mask": inputs_negative["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.float),
            "post_id": post_id,
        }


In [13]:

# Initialize dataset
dataset_train = FineTuneContrastiveDataset(pairs_data_filtered_train, df_posts_filtered_train, df_fact_checks_filtered_train, tokenizer)
dataset_val = FineTuneContrastiveDataset(pairs_data_filtered_val, df_posts_filtered_val, df_fact_checks_filtered_val, tokenizer)

In [26]:
dataset_10 = FineTuneContrastiveDataset(pairs_data_10, posts_mono_sample_1000_10, fact_checks_mono_sample_1000_10, tokenizer)

# Model Fine-Tuning

In [14]:
from transformers import AutoModel
import torch
import torch.nn as nn
import torch.nn.functional as F

class E5FineTuner(nn.Module):
    def __init__(self, model_name, embedding_dim=128):
        """
        Fine-tuneable model for contrastive learning based on pre-trained models.
        Args:
            model_name (str): Name of the pre-trained model from Hugging Face.
            embedding_dim (int): Dimensionality of the projected embedding space.
        """
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.projection = nn.Linear(self.model.config.hidden_size, embedding_dim)  # Projection layer
        self.normalize = nn.LayerNorm(embedding_dim)  # Optional: Normalization for stability

    def forward(self, input_ids, attention_mask):
        """
        Forward pass for encoding input text.
        Args:
            input_ids (torch.Tensor): Tokenized input IDs.
            attention_mask (torch.Tensor): Attention masks for the input IDs.
        Returns:
            torch.Tensor: Normalized embeddings of shape (batch_size, embedding_dim).
        """
        # Get outputs from the transformer
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use pooler_output if available, else mean-pooling over last hidden state
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            embeddings = outputs.pooler_output
        else:
            # Mean-pooling
            embeddings = torch.mean(outputs.last_hidden_state, dim=1)
        
        # Apply projection and normalization
        embeddings = self.projection(embeddings)
        embeddings = self.normalize(embeddings)
        embeddings = F.normalize(embeddings, p=2, dim=1)  # L2-normalization for cosine similarity
        
        return embeddings


# Loss Function

In [15]:
import torch
import torch.nn.functional as F

def contrastive_loss(embeddings, positive_indices, temperature=0.1):
    """
    Compute contrastive loss using InfoNCE formula.

    Args:
        embeddings (torch.Tensor): Tensor of shape (N, D) where N is the batch size and D is the embedding size.
        positive_indices (torch.Tensor): Tensor of shape (N,) indicating the positive pair for each sample.
        temperature (float): Temperature scaling parameter.

    Returns:
        torch.Tensor: Scalar loss.
    """
    # Compute pairwise cosine similarity
    similarity_matrix = F.cosine_similarity(embeddings.unsqueeze(1), embeddings.unsqueeze(0), dim=-1)

    # Scale by temperature
    similarity_matrix /= temperature

    # Create labels for positives
    labels = torch.arange(embeddings.size(0)).to(embeddings.device)
    
    # Use cross-entropy loss
    loss = F.cross_entropy(similarity_matrix, labels)
    
    return loss

# Training Loop

In [31]:
import os
import torch
from collections import deque
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_scheduler
from torch.utils.tensorboard import SummaryWriter
from torch.nn.functional import cosine_similarity

# Inicializa el escritor de TensorBoard
writer = SummaryWriter(log_dir='./runs/e5_fine_tune_infonce_run8')

torch.cuda.empty_cache()

# Hyperparameters
batch_size = 16
epochs = 3
learning_rate = 1e-5
checkpoint_dir = "./checkpoints/run8"
eval_steps = 5000  # Frecuencia de evaluación
best_success_at_10 = 0  # Para rastrear la mejor métrica
best_loss = 0.3  # Para rastrear la mejor pérdida
loss_history = deque(maxlen=1000) 
best_step = 0

# Crear directorio para checkpoints si no existe
os.makedirs(checkpoint_dir, exist_ok=True)

# Prepare DataLoader
dataloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=batch_size)  # Val dataset
val_pairs_df=df_fact_check_post_mapping_filtered_val

# Initialize Model and Optimizer
model = E5FineTuner("intfloat/multilingual-e5-small")
optimizer = AdamW(model.parameters(), lr=learning_rate)

# Scheduler
num_training_steps = len(dataloader) * epochs
num_warmup_steps = int(0.1 * num_training_steps)  # 10% warm-up steps

# Scheduler: Cosine Annealing con LR mínimo
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_training_steps, eta_min=1e-6)

#val_dataloader = DataLoader(validation_dataset, batch_size=batch_size)

# Contrasting loss parameters


# Training Loop
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def success_at_10(predictions_df, pairs_df):
    correct_answers = pairs_df.groupby('post_id')['fact_check_id'].apply(set).to_dict()
    success_scores = []

    for _, row in predictions_df.iterrows():
        post_id = row['post_id']
        predicted_ids = set(int(item['fact_check_id']) for item in row['top_k'])
        success = 1 if post_id in correct_answers and predicted_ids & correct_answers[post_id] else 0
        success_scores.append(success)
    
    predictions_df['success_at_10'] = success_scores
    return predictions_df['success_at_10'].mean()

# Evaluación del modelo
def evaluate_model(model, dataloader, pairs_df):
    model.eval()
    total_loss = 0.0
    all_predictions = []

    with torch.no_grad():
        for batch in dataloader:
            # Mover los datos al dispositivo
            anchor_input_ids = batch["anchor_input_ids"].to(device)
            anchor_attention_mask = batch["anchor_attention_mask"].to(device)
            positive_input_ids = batch["positive_input_ids"].to(device)
            positive_attention_mask = batch["positive_attention_mask"].to(device)
            negative_input_ids = batch["negative_input_ids"].to(device)
            negative_attention_mask = batch["negative_attention_mask"].to(device)

            # Generar embeddings
            anchor_embeddings = model(anchor_input_ids, anchor_attention_mask)
            positive_embeddings = model(positive_input_ids, positive_attention_mask)
            negative_embeddings = model(negative_input_ids, negative_attention_mask)

            # Calcular similitudes coseno y logits
            pos_sim = cosine_similarity(anchor_embeddings, positive_embeddings)
            neg_sim = cosine_similarity(anchor_embeddings, negative_embeddings)
            logits = torch.cat([pos_sim.unsqueeze(1), neg_sim.unsqueeze(1)], dim=1)

            # Calcular pérdida InfoNCE
            labels = torch.zeros(logits.size(0), dtype=torch.long).to(device)
            loss = torch.nn.CrossEntropyLoss()(logits, labels)
            total_loss += loss.item()

            # Recolectar predicciones para Success-at-10
            k = min(10, logits.size(0))  # Esto se ajustará si el número de elementos es menor que 10
            top_k = [{"fact_check_id": str(i)} for i in torch.topk(logits[:, 0], k=k).indices.cpu().numpy()]

            all_predictions.append({"post_id": batch["post_id"], "top_k": top_k})

    # Calcular métricas
    predictions_df = pd.DataFrame(all_predictions)
    success_score = success_at_10(predictions_df, pairs_df)

    return total_loss / len(dataloader), success_score


In [ ]:

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for step, batch in enumerate(dataloader):
        temperature = max(0.1, 0.05 * (1 - step / num_training_steps))

        # Move data to device
        anchor_input_ids = batch["anchor_input_ids"].to(device)
        anchor_attention_mask = batch["anchor_attention_mask"].to(device)
        positive_input_ids = batch["positive_input_ids"].to(device)
        positive_attention_mask = batch["positive_attention_mask"].to(device)
        negative_input_ids = batch["negative_input_ids"].to(device)
        negative_attention_mask = batch["negative_attention_mask"].to(device)

        optimizer.zero_grad()

        # Generate embeddings
        anchor_embeddings = model.forward(anchor_input_ids, anchor_attention_mask)
        positive_embeddings = model.forward(positive_input_ids, positive_attention_mask)
        negative_embeddings = model.forward(negative_input_ids, negative_attention_mask)

        # Calculate cosine similarities
        pos_sim = cosine_similarity(anchor_embeddings, positive_embeddings)
        neg_sim = cosine_similarity(anchor_embeddings, negative_embeddings)

        # Calculate logits
        logits = torch.cat([pos_sim.unsqueeze(1), neg_sim.unsqueeze(1)], dim=1) / temperature
        labels = torch.zeros(logits.size(0), dtype=torch.long).to(device)  # The positive class is always the first

        # Calculate InfoNCE loss
        loss = torch.nn.CrossEntropyLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        # Update metrics
        epoch_loss += loss.item()
        writer.add_scalar('Loss/train', loss.item(), epoch * len(dataloader) + step)
        writer.add_scalar('Learning Rate', scheduler.get_last_lr()[0], epoch * len(dataloader) + step)
        writer.add_scalar('Similarity/positive', pos_sim.mean().item(), step)
        writer.add_scalar('Similarity/negative', neg_sim.mean().item(), step)
        total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        writer.add_scalar('Gradients/total_norm', total_norm, step)

        del anchor_input_ids, anchor_attention_mask, positive_input_ids, positive_attention_mask
        del negative_input_ids, negative_attention_mask, anchor_embeddings, positive_embeddings, negative_embeddings
        torch.cuda.empty_cache()

        new_loss = epoch_loss / (step + 1)  # Avoid division by zero

        if step % 10 == 0:
            print(f"Epoch {epoch}, Step {step}, Loss: {loss.item()}, Total Loss {new_loss}")

        # Calculate the average loss for the last 500 steps
        loss_history.append(loss.item())
        loss_history_list = list(loss_history)
        
        if (step + 1) % eval_steps == 0:
            # Calculate the average loss of the last 500 steps
            avg_loss_last_500 = sum(loss_history_list[-500:-1]) / len(loss_history_list[-500:-1])
            
            # Calculate the average loss of the previous 500 steps
            avg_loss_previous_500 = sum(loss_history_list[-1000:-500]) / len(loss_history_list[-1000:-500]) if len(loss_history_list) > 500 else float('inf')
            
            # If the average loss of the last 500 steps is less than the previous, save the model
            if avg_loss_last_500 < best_loss:
                best_loss = avg_loss_last_500
                checkpoint_path = os.path.join(checkpoint_dir, f"model_step_{step + 1}.pth")
                torch.save(model.state_dict(), checkpoint_path)
                print(f"New model saved at {checkpoint_path} with loss {avg_loss_last_500}")

    # Save checkpoint at the end of each epoch
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': epoch_loss / len(dataloader),
    }, checkpoint_path)
    print(f"Checkpoint saved at {checkpoint_path}")

    # Log epoch metrics
    writer.add_scalar('Loss/epoch', epoch_loss / len(dataloader), epoch)

writer.close()

Epoch 0, Step 0, Loss: 1.0851898193359375, Total Loss 1.0851898193359375
Epoch 0, Step 10, Loss: 0.6241658926010132, Total Loss 0.72465226596052
Epoch 0, Step 20, Loss: 0.34273311495780945, Total Loss 0.6838410553478059
Epoch 0, Step 30, Loss: 0.8283677101135254, Total Loss 0.7157227454646942
Epoch 0, Step 40, Loss: 0.7089816927909851, Total Loss 0.7144508434504997
Epoch 0, Step 50, Loss: 0.5585131049156189, Total Loss 0.7300641338030497
Epoch 0, Step 60, Loss: 0.6316395401954651, Total Loss 0.7242366407738358
Epoch 0, Step 70, Loss: 0.648305356502533, Total Loss 0.7166783742501702
Epoch 0, Step 80, Loss: 0.6938595771789551, Total Loss 0.7076023920082751
Epoch 0, Step 90, Loss: 0.515488862991333, Total Loss 0.6946726736131605
Epoch 0, Step 100, Loss: 0.6243115067481995, Total Loss 0.6939512095829048
Epoch 0, Step 110, Loss: 0.7411436438560486, Total Loss 0.6905443190454362
Epoch 0, Step 120, Loss: 0.689211368560791, Total Loss 0.6888725289628526
Epoch 0, Step 130, Loss: 0.5533662438392

## Large e5 instruct with pre-split data

In [18]:
import os
import torch
from collections import deque
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_scheduler
from torch.utils.tensorboard import SummaryWriter
from torch.nn.functional import cosine_similarity

# Inicializa el escritor de TensorBoard
writer = SummaryWriter(log_dir='./runs/e5_large_instruct_pre_split_data_run1')

torch.cuda.empty_cache()

# Hyperparameters
batch_size = 4
epochs = 3
learning_rate = 1e-5
checkpoint_dir = "./checkpoints/e5_large_instruct_pre_split_data_run1"
eval_steps = 5000  # Frecuencia de evaluación
best_success_at_10 = 0  # Para rastrear la mejor métrica
best_loss = 0.3  # Para rastrear la mejor pérdida
loss_history = deque(maxlen=1000) 
best_step = 0

# Crear directorio para checkpoints si no existe
os.makedirs(checkpoint_dir, exist_ok=True)

# Prepare DataLoader
dataloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(dataset_val, batch_size=batch_size)  # Val dataset
val_pairs_df=pairs_data_filtered_val

# Initialize Model and Optimizer
model = E5FineTuner("intfloat/multilingual-e5-large-instruct")
optimizer = AdamW(model.parameters(), lr=learning_rate)

# Scheduler
num_training_steps = len(dataloader) * epochs
num_warmup_steps = int(0.1 * num_training_steps)  # 10% warm-up steps

# Scheduler: Cosine Annealing con LR mínimo
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_training_steps, eta_min=1e-6)

#val_dataloader = DataLoader(validation_dataset, batch_size=batch_size)

# Contrasting loss parameters


# Training Loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    
model.to(device)

def success_at_10(predictions_df, pairs_df):
    correct_answers = pairs_df.groupby('post_id')['fact_check_id'].apply(set).to_dict()
    success_scores = []

    for _, row in predictions_df.iterrows():
        post_id = row['post_id']
        predicted_ids = set(int(item['fact_check_id']) for item in row['top_k'])
        success = 1 if post_id in correct_answers and predicted_ids & correct_answers[post_id] else 0
        success_scores.append(success)
    
    predictions_df['success_at_10'] = success_scores
    return predictions_df['success_at_10'].mean()

# Evaluación del modelo
def evaluate_model(model, dataloader, pairs_df):
    model.eval()
    total_loss = 0.0
    all_predictions = []

    with torch.no_grad():
        for batch in dataloader:
            # Mover los datos al dispositivo
            anchor_input_ids = batch["anchor_input_ids"].to(device)
            anchor_attention_mask = batch["anchor_attention_mask"].to(device)
            positive_input_ids = batch["positive_input_ids"].to(device)
            positive_attention_mask = batch["positive_attention_mask"].to(device)
            negative_input_ids = batch["negative_input_ids"].to(device)
            negative_attention_mask = batch["negative_attention_mask"].to(device)

            # Generar embeddings
            anchor_embeddings = model(anchor_input_ids, anchor_attention_mask)
            positive_embeddings = model(positive_input_ids, positive_attention_mask)
            negative_embeddings = model(negative_input_ids, negative_attention_mask)

            # Calcular similitudes coseno y logits
            pos_sim = cosine_similarity(anchor_embeddings, positive_embeddings)
            neg_sim = cosine_similarity(anchor_embeddings, negative_embeddings)
            logits = torch.cat([pos_sim.unsqueeze(1), neg_sim.unsqueeze(1)], dim=1)

            # Calcular pérdida InfoNCE
            labels = torch.zeros(logits.size(0), dtype=torch.long).to(device)
            loss = torch.nn.CrossEntropyLoss()(logits, labels)
            total_loss += loss.item()

            # Recolectar predicciones para Success-at-10
            k = min(10, logits.size(0))  # Esto se ajustará si el número de elementos es menor que 10
            top_k = [{"fact_check_id": str(i)} for i in torch.topk(logits[:, 0], k=k).indices.cpu().numpy()]

            all_predictions.append({"post_id": batch["post_id"], "top_k": top_k})

    # Calcular métricas
    predictions_df = pd.DataFrame(all_predictions)
    success_score = success_at_10(predictions_df, pairs_df)

    return total_loss / len(dataloader), success_score


for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for step, batch in enumerate(dataloader):
        temperature = max(0.1, 0.05 * (1 - step / num_training_steps))

        # Move data to device
        anchor_input_ids = batch["anchor_input_ids"].to(device)
        anchor_attention_mask = batch["anchor_attention_mask"].to(device)
        positive_input_ids = batch["positive_input_ids"].to(device)
        positive_attention_mask = batch["positive_attention_mask"].to(device)
        negative_input_ids = batch["negative_input_ids"].to(device)
        negative_attention_mask = batch["negative_attention_mask"].to(device)

        optimizer.zero_grad()

        # Generate embeddings
        anchor_embeddings = model.forward(anchor_input_ids, anchor_attention_mask)
        positive_embeddings = model.forward(positive_input_ids, positive_attention_mask)
        negative_embeddings = model.forward(negative_input_ids, negative_attention_mask)

        # Calculate cosine similarities
        pos_sim = cosine_similarity(anchor_embeddings, positive_embeddings)
        neg_sim = cosine_similarity(anchor_embeddings, negative_embeddings)

        # Calculate logits
        logits = torch.cat([pos_sim.unsqueeze(1), neg_sim.unsqueeze(1)], dim=1) / temperature
        labels = torch.zeros(logits.size(0), dtype=torch.long).to(device)  # The positive class is always the first

        # Calculate InfoNCE loss
        loss = torch.nn.CrossEntropyLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        # Update metrics
        epoch_loss += loss.item()
        writer.add_scalar('Loss/train', loss.item(), epoch * len(dataloader) + step)
        writer.add_scalar('Learning Rate', scheduler.get_last_lr()[0], epoch * len(dataloader) + step)
        writer.add_scalar('Similarity/positive', pos_sim.mean().item(), step)
        writer.add_scalar('Similarity/negative', neg_sim.mean().item(), step)
        total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        writer.add_scalar('Gradients/total_norm', total_norm, step)

        del anchor_input_ids, anchor_attention_mask, positive_input_ids, positive_attention_mask
        del negative_input_ids, negative_attention_mask, anchor_embeddings, positive_embeddings, negative_embeddings
        torch.cuda.empty_cache()

        new_loss = epoch_loss / (step + 1)  # Avoid division by zero

        if step % 10 == 0:
            print(f"Epoch {epoch}, Step {step}, Loss: {loss.item()}, Total Loss {new_loss}")

        # Calculate the average loss for the last 500 steps
        loss_history.append(loss.item())
        loss_history_list = list(loss_history)
        
        if (step + 1) % eval_steps == 0:
            # Calculate the average loss of the last 500 steps
            avg_loss_last_500 = sum(loss_history_list[-500:-1]) / len(loss_history_list[-500:-1])
            
            # Calculate the average loss of the previous 500 steps
            avg_loss_previous_500 = sum(loss_history_list[-1000:-500]) / len(loss_history_list[-1000:-500]) if len(loss_history_list) > 500 else float('inf')
            
            # If the average loss of the last 500 steps is less than the previous, save the model
            if avg_loss_last_500 < best_loss:
                best_loss = avg_loss_last_500
                checkpoint_path = os.path.join(checkpoint_dir, f"model_step_{step + 1}.pth")
                torch.save(model.state_dict(), checkpoint_path)
                print(f"New model saved at {checkpoint_path} with loss {avg_loss_last_500}")

    # Save checkpoint at the end of each epoch
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}.pth")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': epoch_loss / len(dataloader),
    }, checkpoint_path)
    print(f"Checkpoint saved at {checkpoint_path}")

    # Log epoch metrics
    writer.add_scalar('Loss/epoch', epoch_loss / len(dataloader), epoch)

writer.close()

Using device: cuda
GPU: NVIDIA GeForce RTX 3060
Memory Allocated: 4289.17 MB
Epoch 0, Step 0, Loss: 0.6293582916259766, Total Loss 0.6293582916259766


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 12.00 GiB of which 0 bytes is free. Of the allocated memory 17.67 GiB is allocated by PyTorch, and 1.02 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [32]:
correct_answers = df_fact_check_post_mapping_filtered_val.groupby('post_id')['fact_check_id'].apply(set).to_dict()

In [ ]:
val_loss, success_score = evaluate_model(model, val_dataloader, df_fact_check_post_mapping_filtered_val)

In [ ]:
val_loss

In [ ]:
for idx, i in enumerate(val_dataloader):
    if idx >= 10:
        break
    print(i)


In [ ]:
checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch}_e5_fine_tune_infonce_run_all_step_14260.pth")
torch.save({
    'epoch': epoch,
    'step': '14260',
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'loss': epoch_loss / len(dataloader),
}, checkpoint_path)
print(f"Checkpoint saved at {checkpoint_path}")

In [45]:
import os
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_scheduler
from torch.nn.functional import cosine_similarity
from torch.utils.tensorboard import SummaryWriter
import numpy as np

# Inicializa el escritor de TensorBoard
writer = SummaryWriter(log_dir='./runs/e5_fine_tune_infonce_train_val')

torch.cuda.empty_cache()

# Hyperparameters
batch_size = 1
epochs = 3
learning_rate = 1e-5
checkpoint_dir = "./checkpoints/train_val"
eval_steps = 5000  # Frecuencia de evaluación
best_success_at_10 = 0  # Para rastrear la mejor métrica

# Crear directorio para checkpoints si no existe
os.makedirs(checkpoint_dir, exist_ok=True)

# Prepare DataLoader
dataloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader = DataLoader(dataset_val, batch_size=batch_size)  # Val dataset
val_pairs_df=pairs_data_filtered_val

# Initialize Model and Optimizer
model = E5FineTuner("intfloat/multilingual-e5-small")
optimizer = AdamW(model.parameters(), lr=learning_rate)

# Scheduler
num_training_steps = len(dataloader) * epochs
num_warmup_steps = int(0.1 * num_training_steps)  # 10% warm-up steps
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_training_steps, eta_min=1e-6)

# Dispositivo
device ="cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Función para calcular Success-at-10
def success_at_10(predictions_df, pairs_df):
    correct_answers = pairs_df.groupby('post_id')['fact_check_id'].apply(set).to_dict()
    success_scores = []

    for _, row in predictions_df.iterrows():
        post_id = row['post_id']
        predicted_ids = set(int(item['fact_check_id']) for item in row['top_k'])
        success = 1 if post_id in correct_answers and predicted_ids & correct_answers[post_id] else 0
        success_scores.append(success)
    
    predictions_df['success_at_10'] = success_scores
    return predictions_df['success_at_10'].mean()

# Evaluación del modelo
def evaluate_model(model, dataloader, pairs_df):
    model.eval()
    total_loss = 0.0
    all_predictions = []

    with torch.no_grad():
        for batch in dataloader:
            # Mover los datos al dispositivo
            anchor_input_ids = batch["anchor_input_ids"].to(device)
            anchor_attention_mask = batch["anchor_attention_mask"].to(device)
            positive_input_ids = batch["positive_input_ids"].to(device)
            positive_attention_mask = batch["positive_attention_mask"].to(device)
            negative_input_ids = batch["negative_input_ids"].to(device)
            negative_attention_mask = batch["negative_attention_mask"].to(device)

            # Generar embeddings
            anchor_embeddings = model(anchor_input_ids, anchor_attention_mask)
            positive_embeddings = model(positive_input_ids, positive_attention_mask)
            negative_embeddings = model(negative_input_ids, negative_attention_mask)

            # Calcular similitudes coseno y logits
            pos_sim = cosine_similarity(anchor_embeddings, positive_embeddings)
            neg_sim = cosine_similarity(anchor_embeddings, negative_embeddings)
            logits = torch.cat([pos_sim.unsqueeze(1), neg_sim.unsqueeze(1)], dim=1)

            # Calcular pérdida InfoNCE
            labels = torch.zeros(logits.size(0), dtype=torch.long).to(device)
            loss = torch.nn.CrossEntropyLoss()(logits, labels)
            total_loss += loss.item()

            # Recolectar predicciones para Success-at-10
            top_k = [{"fact_check_id": str(i)} for i in torch.topk(logits[:, 0], k=10).indices.cpu().numpy()]
            all_predictions.append({"post_id": batch["post_id"], "top_k": top_k})

    # Calcular métricas
    predictions_df = pd.DataFrame(all_predictions)
    success_score = success_at_10(predictions_df, pairs_df)

    return total_loss / len(dataloader), success_score


In [ ]:

# Training Loop
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0

    for step, batch in enumerate(dataset_train):
        temperature = max(0.1, 0.05 * (1 - step / num_training_steps))

        # Mover los datos al dispositivo
        anchor_input_ids = batch["anchor_input_ids"].to(device)
        anchor_attention_mask = batch["anchor_attention_mask"].to(device)
        positive_input_ids = batch["positive_input_ids"].to(device)
        positive_attention_mask = batch["positive_attention_mask"].to(device)
        negative_input_ids = batch["negative_input_ids"].to(device)
        negative_attention_mask = batch["negative_attention_mask"].to(device)

        anchor_input_ids = anchor_input_ids.unsqueeze(0)  # Agrega una dimensión para el batch
        anchor_attention_mask = anchor_attention_mask.unsqueeze(0)  # Agrega una dimensión para el batch
        positive_input_ids = positive_input_ids.unsqueeze(0)  # Agrega una dimensión para el batch
        positive_attention_mask = positive_attention_mask.unsqueeze(0)  # Agrega una dimensión para el batch
        
        negative_input_ids = negative_input_ids.unsqueeze(0)  # Agrega una dimensión para el batch
        negative_attention_mask = negative_attention_mask.unsqueeze(0)  # Agrega una dimensión para el batch

        optimizer.zero_grad()

        # Generar embeddings
        anchor_embeddings = model(anchor_input_ids, anchor_attention_mask)
        positive_embeddings = model(positive_input_ids, positive_attention_mask)
        negative_embeddings = model(negative_input_ids, negative_attention_mask)

        # Calcular logits
        pos_sim = cosine_similarity(anchor_embeddings, positive_embeddings)
        neg_sim = cosine_similarity(anchor_embeddings, negative_embeddings)
        logits = torch.cat([pos_sim.unsqueeze(1), neg_sim.unsqueeze(1)], dim=1) / temperature
        labels = torch.zeros(logits.size(0), dtype=torch.long).to(device)

        # Calcular la pérdida InfoNCE
        loss = torch.nn.CrossEntropyLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        # Actualizar métricas
        epoch_loss += loss.item()
        writer.add_scalar('Loss/train', loss.item(), epoch * len(dataloader) + step)

        if (step + 1) % eval_steps == 0:
            print(f"Evaluating at step {step + 1}...")
            val_loss, success_score = evaluate_model(model, val_dataloader, val_pairs_df)
            writer.add_scalar('Loss/val', val_loss, step + 1)
            writer.add_scalar('Success-at-10/val', success_score, step + 1)
            print(f"Step {step + 1}: Val Loss = {val_loss:.4f}, Success-at-10 = {success_score:.4f}")

            # Guardar si la métrica mejora
            if success_score > best_success_at_10:
                best_success_at_10 = success_score
                checkpoint_path = os.path.join(checkpoint_dir, f"best_model_step_{step + 1}.pth")
                torch.save(model.state_dict(), checkpoint_path)
                print(f"New best model saved at {checkpoint_path}")

        del anchor_embeddings, positive_embeddings, negative_embeddings, logits
        torch.cuda.empty_cache() 


    writer.add_scalar('Loss/epoch', epoch_loss / len(dataloader), epoch)

writer.close()


In [ ]:
import tracemalloc

# Iniciar el rastreo de memoria
tracemalloc.start()

# Tu código que quieres monitorear
# (Coloca esto antes de las partes sospechosas)
# Por ejemplo:
# anchor_embeddings = model(anchor_input_ids, anchor_attention_mask)

# Tomar una instantánea después de ejecutar una parte del código
snapshot = tracemalloc.take_snapshot()

# Mostrar las 10 líneas con mayor consumo de memoria
top_stats = snapshot.statistics('lineno')

print("[ Top 10 consumidores de memoria ]")
for stat in top_stats[:10]:
    print(stat)

In [ ]:
with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU
    ],
    on_trace_ready=torch.profiler.tensorboard_trace_handler('./log'),
    record_shapes=True
) as prof:
    # Código del modelo
    prof.export_chrome_trace("trace.json")

# Index of troubled steps

In [ ]:
# Parámetros del DataLoader
batch_size = dataloader.batch_size

# Step que quieres rastrear
step = 1100

# Calcular los índices correspondientes en el dataset
start_idx = step * batch_size
end_idx = start_idx + batch_size

print(f"Índices en el dataset para el step {step}: {start_idx} a {end_idx}")

In [ ]:
# Extraer datos del dataset original
step_data = dataloader.dataset[start_idx:end_idx]

# Inspeccionar
for idx, data in enumerate(step_data):
    print(f"Dato {start_idx + idx}: {data}")

# Busqueda de outliers

In [ ]:
text_lengths = [len(text.split()) for text in df_posts["ocr_text"]]

import matplotlib.pyplot as plt
plt.hist(text_lengths, bins=50)
plt.title("Distribución de la longitud de texto")
plt.xlabel("Número de tokens")
plt.ylabel("Frecuencia")
plt.show()

# Detectar outliers
import numpy as np
mean_length = np.mean(text_lengths)
std_length = np.std(text_lengths)
outlier_threshold = mean_length + 3 * std_length

outliers = [i for i, length in enumerate(text_lengths) if length > outlier_threshold]
print(f"Textos outliers en longitud: {outliers}")

In [ ]:
# Count the number of text lengths greater than 1000
num_text_lengths_above_1000 = sum(length > 1000 for length in text_lengths)
print(f"Number of text lengths greater than 1000: {num_text_lengths_above_1000}")

In [ ]:
text_lengths = [len(text.split()) for text in df_fact_checks["claim_title"]]

import matplotlib.pyplot as plt
plt.hist(text_lengths, bins=50)
plt.title("Distribución de la longitud de texto")
plt.xlabel("Número de tokens")
plt.ylabel("Frecuencia")
plt.show()

# Detectar outliers
import numpy as np
mean_length = np.mean(text_lengths)
std_length = np.std(text_lengths)
outlier_threshold = mean_length + 3 * std_length

outliers = [i for i, length in enumerate(text_lengths) if length > outlier_threshold]
print(f"Textos outliers en longitud: {outliers}")

In [ ]:
# Count the number of text lengths greater than 1000
num_text_lengths_above_1000 = sum(length > 100 for length in text_lengths)
print(f"Number of text lengths greater than 1000: {num_text_lengths_above_1000}")

In [ ]:
df_fact_checks[df_fact_checks['claim_title'].str.len() == 0]

In [ ]:
len(df_fact_checks.index.unique()), len(df_fact_checks)

In [ ]:
df_fact_check_post_mapping['fact_check_id'].nunique()

In [ ]:
print(df_fact_checks_filtered_train[df_fact_checks_filtered_train["claim_title"].isnull()])  # Filas con NaN
print(df_fact_checks_filtered_train[df_fact_checks_filtered_train["claim_title"] == ""])    # Filas con cadenas vacías

In [ ]:
pairs_data_filtered_train["o